<a href="https://colab.research.google.com/github/BrianAyekha/-connection-of-MySQL-database-to-NodeJs/blob/main/Week2_Data_and_ML_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Assignment | Data and ML Workflow
**Project:** Early fault detection and diagnosis in hospital electromechanical equipment (infant warmer case study)

**Author:** Brian Ayekha Mwangala

## 1. Project problem and intended AI/ML outcome

**Problem:** Faults in hospital electromechanical equipment, such as a failing heater or SSR, a degrading power supply or a drifting temperature sensor, usually show up only after the device stops working or reads wrongly. Infant warmers are life-supporting devices, so a silent fault puts a newborn at risk of hypothermia or burns. In many Kenyan hospitals, technicians are few and equipment is old, so diagnosis is slow and reactive.

**Intended AI/ML outcome:** Use sensor and operating data to
1. **detect** when a machine is about to fail or has failed (binary classification), and
2. **classify** the likely fault type (multi-class / multi-label classification), so that maintenance can be done before the device fails.

Predicting remaining time before failure is a possible later extension. I am not yet sure whether enough data exists for it.

## 2. Data I would need

**Ideal data (for infant warmers):**
- Temperature over time: skin probe, air and heater element
- Heater duty cycle, SSR switching behaviour, supply voltage and current
- Alarm event logs
- Maintenance records: fault type, component replaced, device age, usage hours
- Ambient conditions and mains quality

**Problem:** I could not find a public dataset of infant warmer faults. I have not yet logged enough labelled fault data from my own Arduino warmer prototype or from service work.

**Dataset used instead: AI4I 2020 Predictive Maintenance Dataset** (UCI Machine Learning Repository, id 601; also on Kaggle).
It is a synthetic dataset of 10,000 rows modelled on a milling machine. It has sensor readings and failure labels, so it is a good stand-in for this project's problem: detect and classify faults from sensor behaviour.

| AI4I column | Closest analogue in an infant warmer |
|---|---|
| Air temperature | Ambient / air temperature |
| Process temperature | Heater element or mattress temperature |
| Rotational speed, Torque | Load on the system (e.g. supply current/voltage) |
| Tool wear | Operating hours / component ageing |
| HDF (heat dissipation failure) | Overheating or poor heat transfer |
| PWF (power failure) | Power supply fault |

**Limitations:** The data is synthetic and not from medical equipment. The failure rate is only about 3.4%, so the classes are heavily imbalanced. Results will show the workflow, but they will not prove the method works on real warmers.

*Citation:* S. Matzka, "Explainable Artificial Intelligence for Predictive Maintenance Applications," 2020 Third International Conference on Artificial Intelligence for Industries (AI4I), pp. 69-74. doi: 10.1109/AI4I49448.2020.00023

## 3. Import the dataset

In [1]:
# Install and import the libraries
!pip install -q ucimlrepo

import pandas as pd
from ucimlrepo import fetch_ucirepo

In [2]:
# Download the AI4I 2020 dataset directly from the UCI repository (id = 601)
ai4i = fetch_ucirepo(id=601)

# ucimlrepo returns the inputs (X) and the targets (y) as two pandas DataFrames
X_raw = ai4i.data.features
y_raw = ai4i.data.targets

# Join them into one table so we can look at the whole dataset
df = pd.concat([X_raw, y_raw], axis=1)
print("Rows, columns:", df.shape)

Rows, columns: (10000, 12)


## 4. Display a small portion of the data

In [3]:
# First 5 rows
df.head()

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
# Data types, missing values, and summary statistics
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Type                 10000 non-null  object 
 1   Air temperature      10000 non-null  float64
 2   Process temperature  10000 non-null  float64
 3   Rotational speed     10000 non-null  int64  
 4   Torque               10000 non-null  float64
 5   Tool wear            10000 non-null  int64  
 6   Machine failure      10000 non-null  int64  
 7   TWF                  10000 non-null  int64  
 8   HDF                  10000 non-null  int64  
 9   PWF                  10000 non-null  int64  
 10  OSF                  10000 non-null  int64  
 11  RNF                  10000 non-null  int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 937.6+ KB


,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,300.004930,310.005560,1538.776100,39.986910,107.951000,0.033900,0.004600,0.011500,0.009500,0.009800,0.00190
std,2.000259,1.483734,179.284096,9.968934,63.654147,0.180981,0.067671,0.106625,0.097009,0.098514,0.04355
min,295.300000,305.700000,1168.000000,3.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
25%,298.300000,308.800000,1423.000000,33.200000,53.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
50%,300.100000,310.100000,1503.000000,40.100000,108.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
75%,301.500000,311.100000,1612.000000,46.800000,162.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000
max,304.500000,313.800000,2886.000000,76.600000,253.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000


In [5]:
# How many failures are there? (class balance)
print(df["Machine failure"].value_counts())
print("\nFailure rate: {:.2f}%".format(df["Machine failure"].mean() * 100))

# Count of each failure mode (TWF, HDF, PWF, OSF, RNF)
df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum()

Machine failure
0    9661
1     339
Name: count, dtype: int64

Failure rate: 3.39%


,0
TWF,46
HDF,115
PWF,95
OSF,98
RNF,19


## 5. Inputs (features) and output (target)

**Inputs (features)**
- `Type`: product quality class (L, M or H). It is categorical, so it will need encoding.
- `Air temperature` (K)
- `Process temperature` (K)
- `Rotational speed` (rpm)
- `Torque` (Nm)
- `Tool wear` (min)

**Outputs (targets)**
- Main target: `Machine failure` (0 = normal, 1 = failure). This is binary classification.
- Secondary target: the failure mode columns `TWF`, `HDF`, `PWF`, `OSF`, `RNF`. This is for fault-type classification.

**Not used as inputs:** `UID` and `Product ID` are only identifiers. The failure-mode columns must not be used as inputs when predicting `Machine failure`, because they would leak the answer.

**Possible later feature:** the difference between process and air temperature. The dataset documentation says heat dissipation failure depends on it, so it is probably useful.

In [6]:
# Separate the inputs (X) from the target (y)
feature_cols = ["Type", "Air temperature", "Process temperature",
                "Rotational speed", "Torque", "Tool wear"]

X = df[feature_cols]
y = df["Machine failure"]            # main target (binary)
y_modes = df[["TWF", "HDF", "PWF", "OSF", "RNF"]]   # secondary target (fault type)

print("Features:", list(X.columns))
print("Target: Machine failure")
X.head()

Features: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear']
Target: Machine failure


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


## 6. Proposed ML workflow

```
 ┌──────────────────────┐
 │ 1. Define problem    │  Detect failure early + classify fault type
 └──────────┬───────────┘
            ▼
 ┌──────────────────────┐
 │ 2. Collect data      │  AI4I 2020 now; later my own warmer
 └──────────┬───────────┘  prototype logs and service records
            ▼
 ┌──────────────────────┐
 │ 3. Explore data      │  Distributions, correlations,
 └──────────┬───────────┘  class imbalance (~3.4% failures)
            ▼
 ┌──────────────────────┐
 │ 4. Preprocess        │  Encode Type, scale numeric features,
 └──────────┬───────────┘  add engineered features
            ▼
 ┌──────────────────────┐
 │ 5. Split data        │  Train / test (stratified)
 └──────────┬───────────┘
            ▼
 ┌──────────────────────┐
 │ 6. Train models      │  Baseline first, then stronger models;
 └──────────┬───────────┘  handle imbalance
            ▼
 ┌──────────────────────┐
 │ 7. Evaluate          │  Recall, precision, F1, confusion matrix
 └──────────┬───────────┘  (missed failures matter most)
            ▼
 ┌──────────────────────┐
 │ 8. Improve / iterate │  Tune, add features, check errors
 └──────────┬───────────┘
            ▼
 ┌──────────────────────┐
 │ 9. Deploy / test     │  Try on real warmer sensor data
 └──────────────────────┘
```

**Brief explanation**
1. **Define the problem:** state what to predict and why, as in Section 1.
2. **Collect data:** use the public dataset now, and gather real warmer data later.
3. **Explore:** check the data for patterns, missing values and class imbalance.
4. **Preprocess:** convert categories to numbers, scale values, and create useful features.
5. **Split:** keep a test set aside so the model is judged on data it has not seen.
6. **Train:** start with a simple baseline, then try stronger models.
7. **Evaluate:** accuracy alone is misleading with 3.4% failures, so I will focus on recall (missed failures) and precision (false alarms).
8. **Iterate:** improve the model based on its errors.
9. **Deploy/test:** check how the model behaves on real infant warmer data. This step depends on data I do not have yet.

I have not yet decided which model to use.